# Vectorbt rewrite

This notebook replaces the backtesting.py workflow with a vectorbt-based EMA/VWAP backtest.

In [7]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys
sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free

CASH = 80_000
COMMISSION = 0.00015
RISK_PER_TRADE = 0.001

def load_data(file_path, *, verbose=False):
    # Compact float32 + Parquet-cached loader; attaches a daily-reset VWAP column.
    return load_ohlcv(file_path, use_cache=True, add_vwap=True, verbose=verbose)

def make_signals(data, n1=9, n2=21, atr_period=14, sl_mult=1.0, rr_ratio=0.5):
    # keep float32 through indicator math; cast up only at the vectorbt boundary
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    ema2 = vbt.indicators.MA.run(close, window=n2, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close
    size_value = (CASH * RISK_PER_TRADE) / sl_pct

    long_signal = (ema1.ma > ema2.ma) & (close > data["VWAP"]) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_signal = (ema2.ma > ema1.ma) & (close < data["VWAP"]) & np.isfinite(atr.atr) & (atr.atr > 0)

    return pd.DataFrame({
        "ema1": ema1.ma,
        "ema2": ema2.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "size_value": size_value.where(np.isfinite(size_value) & (size_value > 0), 0.0),
        "long": long_signal,
        "short": short_signal,
    })

def run_backtest(data, n1=9, n2=21, atr_period=14, sl_mult=1.0, rr_ratio=0.5):
    signals = make_signals(data, n1=n1, n2=n2, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy()
    short_signal = signals["short"].to_numpy()
    # vbt wants float64 for size/close; cast just the columns it needs.
    close_np = data["Close"].to_numpy(dtype=np.float64)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=short_signal,
        short_entries=short_signal,
        short_exits=long_signal,
        init_cash=CASH,
        fees=COMMISSION,
        size=signals["size_value"].to_numpy(dtype=np.float64),
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=signals["tp_pct"].to_numpy(dtype=np.float64),
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, n2, atr_period, sl_mult, rr_ratio):
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, n2=n2, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio)
        return {
            "n1": n1, "n2": n2, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "n2": n2, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [8]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M1.csv").expanduser()
data = load_data(file_path, verbose=True)

portfolio, stats, signals = run_backtest(data, rr_ratio=1)
print(stats)
# portfolio.plot()
trades = portfolio.trades.records_readable
print(trades)
free(data, signals, portfolio, trades)

[loader] cache hit: XAUUSD_M1__cache_v1.parquet
Start                                                 0
End                                             2088752
Period                               1450 days 12:33:00
Start Value                                     80000.0
End Value                                   -388.719137
Total Return [%]                            -100.485899
Benchmark Return [%]                         184.185378
Max Gross Exposure [%]                 159093722.942569
Total Fees Paid                            79880.486415
Max Drawdown [%]                             100.511606
Max Drawdown Duration                1450 days 12:05:00
Total Trades                                       1350
Total Closed Trades                                1349
Total Open Trades                                     1
Open Trade PnL                              -248.918289
Win Rate [%]                                  27.946627
Best Trade [%]                                  1.13837


In [ ]:
db_path = Path("backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS v1_rr_vectorbt (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    cagr_pct REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

In [ ]:
asset = "XAUUSD"
timeframe = "M5"
file_path = Path(f"~/market_data/{asset}/{asset}_{timeframe}.csv").expanduser()
data = load_data(file_path)
portfolio, stats, signals = run_backtest(data)

row = (
    asset,
    timeframe,
    9,
    21,
    float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
    float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
    float(stats["CAGR [%]"]) if stats.get("CAGR [%]") is not None else None,
    float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
    float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
    float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
    int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
    float(stats["End Value"]) if stats.get("End Value") is not None else None,
)

cursor.execute(
    "INSERT INTO v1_rr_vectorbt (asset, timeframe, n1, n2, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    row,
)
conn.commit()
stats

## Optimization

In [ ]:

from itertools import product
import time


TABLE_NAME = "test"
db_path = Path("backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15), # fast ema
    range(18, 30), # slow ema
    range(10, 21), # atr period
    [0.5, 1.0, 1.5, 2.0], # sl mult
    [0.25, 0.5, 0.75, 1.0, 1.5, 2.0], # rr ratio
))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
# Load each timeframe once and reuse across all param combos (big memory win on M1).
data_by_tf: dict[str, pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
    data_by_tf[tf] = load_data(fp, verbose=True)

for i, (timeframe, n1, n2, ap, sm, rr) in enumerate(grid):

    data = data_by_tf[timeframe]

    t1 = time.time()
    result = run_params(data, n1, n2, ap, sm, rr)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, n2, atr_period, sl_mult, rr_ratio, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["n2"], result["atr_period"], result["sl_mult"], result["rr_ratio"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    print(f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} n2={n2} ap={ap} sm={sm} rr={rr} | sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s)")
    if (i + 1) % 100 == 0:
        print(f"[{time.strftime('%H:%M:%S')}] {i+1}/{len(grid)} done in {time.time()-t0:.1f}s")
    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)

In [10]:

from itertools import product
import time


TABLE_NAME = "test_paired"
db_path = Path("backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15), # fast ema
    range(18, 30), # slow ema
    [14], # atr period
    [1], # sl mult
    [1], # rr ratio
))

t0 = time.time()
print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
data_by_tf: dict[str, pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
    data_by_tf[tf] = load_data(fp, verbose=True)

for i, (timeframe, n1, n2, ap, sm, rr) in enumerate(grid):

    data = data_by_tf[timeframe]

    result = run_params(data, n1, n2, ap, sm, rr)
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, n2, atr_period, sl_mult, rr_ratio, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["n2"], result["atr_period"], result["sl_mult"], result["rr_ratio"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    if (i + 1) % 100 == 0:
        print(f" {i+1}/{len(grid)} done in {time.time()-t0:.1f}s")
    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df[sweep_df["num_trades"] >= 1000]
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null & trades>=1000: {len(sweep_df)}")
sweep_df.head(20)

pair_groups = sweep_df.groupby(["n1", "n2"]).agg(
    sharpe_mean=("sharpe_ratio", "mean"),
    expectancy_mean=("expectancy", "mean"),
    return_pct_mean=("return_pct", "mean"),
    max_dd_mean=("max_dd", "mean"),
    win_rate_mean=("win_rate", "mean"),
    profit_factor_mean=("profit_factor", "mean"),
    trades_mean=("num_trades", "mean"),
    count=("sharpe_ratio", "size"),
).reset_index()

best_sharpe = pair_groups.sort_values("sharpe_mean", ascending=False).head(10)
print("\nTop pairs by Sharpe Ratio (mean across timeframes):")
print(best_sharpe)

best_expectancy = pair_groups.sort_values("expectancy_mean", ascending=False).head(10)
print("\nTop pairs by Expectancy (mean across timeframes):")
print(best_expectancy)

pair_groups["sharpe_rank"] = pair_groups["sharpe_mean"].rank(ascending=False)
pair_groups["expectancy_rank"] = pair_groups["expectancy_mean"].rank(ascending=False)
pair_groups["combined_rank"] = pair_groups["sharpe_rank"] + pair_groups["expectancy_rank"]
best_combined = pair_groups.sort_values("combined_rank").head(10)
print("\nTop pairs by combined Sharpe + Expectancy rank:")
print(best_combined)


[11:01:43] Testing 360 combinations -> test_paired
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] parsing CSV: XAUUSD_M15.csv
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
 100/360 done in 85.3s
 200/360 done in 132.1s
 300/360 done in 164.8s
Total grid: 360 | Non-null & trades>=1000: 360

Top pairs by Sharpe Ratio (mean across timeframes):
     n1  n2  sharpe_mean  expectancy_mean  return_pct_mean  max_dd_mean  \
16    6  22     0.008540       -31.384397      -100.358686   100.375675   
26    7  20    -0.106723       -31.047055      -100.340866   100.358011   
38    8  20    -0.271055       -31.231268      -100.307159   100.321986   
7     5  25    -0.299765       -31.329241      -100.244750   100.255637   
94   12  28    -0.339608       -30.874707      -100.244975   100.257041   
33    7  27    -0.345882       -31.371868      -100.230816   100.240737   
25    7  19    -0.347257       -31.327441      -100.160232   100.200334   
108  14  18    -0.365547       -31.022116    